# Lanzamiento de un cohete.

El problema obligatorio presente consiste en el lanzamiento de un cohete para que orbite alrededor de la Luna.

## Código.

Código final de la simulación utilizado:
```cpp

#include <array>
#include <cmath>
#include <fstream>
#include <iostream>
#include <vector>

using namespace std;

// Constantes físicas (SI)
const double G = 6.67e-11;
const double MT = 5.9736e24;
const double ML = 0.07349e24;
const double dTL = 3.844e8;
const double omega = 2.6617e-6; // Velocidad angular de la Luna en torno a la Tierra

// Parámetros reescalados
const double delta = G * MT / pow(dTL, 3);
const double mu = ML / MT;

// Declaración de funciones existentes
void f(double t, const double y[], double dydt[]);
void paso_RK4(double t, double h, double y[], int n, int N);
void integracion_adaptativa(double &t, double &h, double y[], int N);

// NUEVA FUNCIÓN: Calcula la constante de movimiento H' = H - omega * p_phi (adimensional)
double calcular_H_prima(const double y[], double t) {
    double r_tilde = y[0];
    double phi = y[1];
    double pr_tilde = y[2];
    double pphi_tilde = y[3];
    
    double r_prime = sqrt(r_tilde * r_tilde + 1.0 - 2.0 * r_tilde * cos(phi - omega * t));
    
    // Energía total menos el término de rotación
    double H_prima = 0.5 * (pr_tilde * pr_tilde + (pphi_tilde * pphi_tilde) / (r_tilde * r_tilde)) 
                     - delta * (1.0 / r_tilde + mu / r_prime) 
                     - omega * pphi_tilde;
    return H_prima;
}

int main() {
// Condiciones iniciales (NUEVAS: Órbita terrestre a 500 km)
    double altitud = 500e3; // 500 km sobre la superficie
    double r_real = 6.37816e6 + altitud;
    double r0 = r_real / dTL; // Distancia inicial normalizada
    
    // Velocidad orbital (menor a la de escape para quedar atrapado)
    double v0 = 8000.0; // m/s
    double phi0 = 0.0; 
    
    // Momentos adimensionales corregidos físicamente
    double pr0_tilde = 0.0; 
    double pphi0_tilde = r0 * (v0 / dTL); 

    // Guardamos el estado inicial para ambas simulaciones
    double y_inicial[4] = {r0, phi0, pr0_tilde, pphi0_tilde};
    
    // Calculamos el valor analítico constante de H' en t=0
    double H_prima_0 = calcular_H_prima(y_inicial, 0.0);

    // =======================================================
    // PARTE 1: SIMULACIÓN CON PASO FIJO
    // =======================================================
    double y_fijo[4] = {y_inicial[0], y_inicial[1], y_inicial[2], y_inicial[3]};
    double t_fijo = 0.0;
    double h_fijo = 30.0; // 1 minuto (sugerido por el problema)
    int iteraciones_fijo = 0;
    
    ofstream archivo_fijo("trayectoria_fija.dat");
    if (!archivo_fijo.is_open()) return 1;

    cout << "Iniciando simulacion con paso FIJO (h = " << h_fijo << " s)..." << endl;
    
    bool continuar_fijo = true;
    while (continuar_fijo) {
        paso_RK4(t_fijo, h_fijo, y_fijo, 4, 1);
        t_fijo += h_fijo; // Necesario sumar manualmente porque paso_RK4 toma 't' por valor
        iteraciones_fijo++;

        double r_tilde = y_fijo[0];
        double phi = y_fijo[1];
        double r_prime = sqrt(r_tilde * r_tilde + 1.0 - 2.0 * r_tilde * cos(phi - omega * t_fijo));

        // Guardamos puntos cada 100 iteraciones para no generar archivos inmensos
        if (iteraciones_fijo % 100 == 0) {
            double x_nave = r_tilde * cos(phi);
            double y_nave = r_tilde * sin(phi);
            double x_luna = cos(omega * t_fijo);
            double y_luna = sin(omega * t_fijo);
            archivo_fijo << x_nave << "," << y_nave << "\n" << x_luna << "," << y_luna << "\n\n";
        }

        // Condiciones de parada originales
        if (t_fijo > 100.0 && r_tilde * dTL < 6.37816e6) continuar_fijo = false;
        if (r_prime * dTL < 1.7374e6) continuar_fijo = false;
        if (t_fijo > 864000.0) continuar_fijo = false;
    }
    archivo_fijo.close();
    
    // Calculamos el error acumulado de la constante H'
    double error_H_fijo = fabs(calcular_H_prima(y_fijo, t_fijo) - H_prima_0);


    // =======================================================
    // PARTE 2: SIMULACIÓN CON PASO ADAPTATIVO
    // =======================================================
    double y_adap[4] = {y_inicial[0], y_inicial[1], y_inicial[2], y_inicial[3]};
    double t_adap = 0.0;
    double h_adap = 30.0; // Paso inicial sugerido
    int iteraciones_adap = 0;
    
    ofstream archivo_adap("trayectoria_adap.dat");
    if (!archivo_adap.is_open()) return 1;

    cout << "Iniciando simulacion con paso ADAPTATIVO..." << endl;
    
    bool continuar_adap = true;
    while (continuar_adap) {
        integracion_adaptativa(t_adap, h_adap, y_adap, 1);
        iteraciones_adap++; // La función 'integracion_adaptativa' sí actualiza 't_adap' por referencia

        double r_tilde = y_adap[0];
        double phi = y_adap[1];
        double r_prime = sqrt(r_tilde * r_tilde + 1.0 - 2.0 * r_tilde * cos(phi - omega * t_adap));

        if (iteraciones_adap % 100 == 0) {
            double x_nave = r_tilde * cos(phi);
            double y_nave = r_tilde * sin(phi);
            double x_luna = cos(omega * t_adap);
            double y_luna = sin(omega * t_adap);
            archivo_adap << x_nave << "," << y_nave << "\n" << x_luna << "," << y_luna << "\n\n";
        }

        if (t_adap > 100.0 && r_tilde * dTL < 6.37816e6) continuar_adap = false;
        if (r_prime * dTL < 1.7374e6) continuar_adap = false;
        if (t_adap > 864000.0) continuar_adap = false;
    }
    archivo_adap.close();
    
    double error_H_adap = fabs(calcular_H_prima(y_adap, t_adap) - H_prima_0);


    // =======================================================
    // RESULTADOS FINALES Y COMPARACIÓN OBLIGATORIA
    // =======================================================
    cout << "\n==============================================" << endl;
    cout << "  COMPARACION: H FIJA vs H ADAPTATIVA" << endl;
    cout << "==============================================" << endl;
    cout << "METODO FIJO:" << endl;
    cout << "  - Iteraciones (esfuerzo): " << iteraciones_fijo << endl;
    cout << "  - Error acumulado en H':  " << error_H_fijo << endl;
    cout << "\nMETODO ADAPTATIVO:" << endl;
    cout << "  - Iteraciones (esfuerzo): " << iteraciones_adap << endl;
    cout << "  - Error acumulado en H':  " << error_H_adap << endl;
    cout << "==============================================" << endl;
    cout << "Archivos 'trayectoria_fija.dat' y 'trayectoria_adap.dat' generados con exito." << endl;

    return 0;
}

// [MANTENER AQUÍ TUS FUNCIONES ORIGINALES: f, paso_RK4 e integracion_adaptativa]
void f(double t, const double y[], double dydt[]) {
    double r_tilde = y[0];
    double phi = y[1];
    double pr_tilde = y[2];
    double pphi_tilde = y[3];

    double r_prime = sqrt(r_tilde * r_tilde + 1.0 - 2.0 * r_tilde * cos(phi - omega * t));

    dydt[0] = pr_tilde;
    dydt[1] = pphi_tilde / (r_tilde * r_tilde);

    dydt[2] = (pphi_tilde * pphi_tilde) / (r_tilde * r_tilde * r_tilde)
              - delta * (1.0 / (r_tilde * r_tilde)
              + (mu / (r_prime * r_prime * r_prime)) * (r_tilde - cos(phi - omega * t)));

    dydt[3] = - (delta * mu * r_tilde / (r_prime * r_prime * r_prime)) * sin(phi - omega * t);
}

void paso_RK4(double t, double h, double y[], int n, int N) {
    double k1[4], k2[4], k3[4], k4[4], aux[4];

    for (int step = 0; step < N; step++) {
        f(t, y, k1);
        for (int i = 0; i < n; i++) aux[i] = y[i] + 0.5 * h * k1[i];

        f(t + 0.5 * h, aux, k2);
        for (int i = 0; i < n; i++) aux[i] = y[i] + 0.5 * h * k2[i];

        f(t + 0.5 * h, aux, k3);
        for (int i = 0; i < n; i++) aux[i] = y[i] + h * k3[i];

        f(t + h, aux, k4);
        for (int i = 0; i < n; i++) {
            y[i] += (h / 6.0) * (k1[i] + 2.0 * k2[i] + 2.0 * k3[i] + k4[i]);
        }
        
        t += h; 
    }
}

void integracion_adaptativa(double &t, double &h, double y[], int N) {
    double y_h[4], y_h2[4];
    double epsilon_max = 1e-12;

    for (int i = 0; i < 4; i++) {
        y_h[i] = y_h2[i] = y[i];
    }

    paso_RK4(t, h, y_h, 4, N);
    paso_RK4(t, h / 2.0, y_h2, 4, N);
    paso_RK4(t + h / 2.0, h / 2.0, y_h2, 4, N);

    double error = 0.0;
    for (int i = 0; i < 4; i++) {
        double e_i = (16.0 / 15.0) * fabs(y_h2[i] - y_h[i]);
        if (e_i > error) error = e_i;
    }

    double s = pow(error / epsilon_max, 0.2);
    if (s > 2.0) {
        h = h / 2.0;
    } else {
        t += h;
        for (int i = 0; i < 4; i++) y[i] = y_h2[i];
        if (s < 0.1) h = 2.0 * h;
    }
}

```

Código utilizado para la animación, proporcionado por la profesora Jara Juana Bermejo Vega:

In [ ]:
# ================================================================================
# ANIMACION SISTEMA SOLAR
#
# Genera una animación a partir de un fichero de datos con las posiciones
# de los planetas en diferentes instantes de tiempo.
# 
# El fichero debe estructurarse de la siguiente forma:
# 
#   x1_1, y1_1
#   x2_1, y2_1
#   x3_1, y3_1
#   (...)
#   xN_1, yN_1
#   
#   x1_2, y1_2
#   x2_2, y2_2
#   x3_2, y3_2
#   (...)
#   xN_2, yN_2
#
#   x1_3, y1_3
#   x2_3, y2_3
#   x3_3, y3_3
#   (...)
#   xN_3, yN_3
#   
#   (...)
#
# donde xi_j es la componente x del planeta i-ésimo en el instante de
# tiempo j-ésimo, e yi_j lo mismo en la componente y. El programa asume que
# el nº de planetas es siempre el mismo.
# ¡OJO! Los datos están separados por comas.
# 
# Si solo se especifica un instante de tiempo, se genera una imagen en pdf
# en lugar de una animación
#
# Se puede configurar la animación cambiando el valor de las variables
# de la sección "Parámetros"
#
# ================================================================================

# Importa los módulos necesarios
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Circle
import numpy as np

# Parámetros
# ========================================
file_in = "trayectoria_fija.dat" # Nombre del fichero de datos
file_out = "cohete" # Nombre del fichero de salida (sin extensión)

# Límites de los ejes X e Y
x_min = -5
x_max = 5
y_min = -5
y_max = 5

interval = 10 # Tiempo entre fotogramas en milisegundos
show_trail = True # Muestra la "estela" del planeta
trail_width = 1 # Ancho de la estela
save_to_file = True # False: muestra la animación por pantalla,
                     # True: la guarda en un fichero
dpi = 150 # Calidad del vídeo de salida (dots per inch)

# Radio del planeta, en las mismas unidades que la posición
# Puede ser un número (el radio de todos los planetas) o una lista con
# el radio de cada uno
planet_radius = 0.05 
# planet_radius = [0.1, 0.1, 0.2, 0.2, 1.5, 1.3, 1, 1, 0.1]

# Colores de cada planeta. Si solo se especifica un color, se usa el mismo
# para todos los planetas.
planet_colors = ["tab:blue", "tab:gray"]


# Lectura del fichero de datos
# ========================================
# Lee el fichero a una cadena de texto
with open(file_in, "r") as f:
    data_str = f.read()

# Inicializa la lista con los datos de cada fotograma.
# frames_data[j] contiene los datos del fotograma j-ésimo
frames_data = list()

# Itera sobre los bloques de texto separados por líneas vacías
# (cada bloque corresponde a un instante de tiempo)
for frame_data_str in data_str.split("\n\n"):
    # Inicializa la lista con la posición de cada planeta
    frame_data = list()

    # Itera sobre las líneas del bloque
    # (cada línea da la posición de un planta)
    for planet_pos_str in frame_data_str.split("\n"):
        # Lee la componente x e y de la línea
        planet_pos = np.fromstring(planet_pos_str, sep=",")
        # Si la línea no está vacía, añade planet_pos a la lista de 
        # posiciones del fotograma
        if planet_pos.size > 0:
            frame_data.append(np.fromstring(planet_pos_str, sep=","))

    # Añade los datos de este fotograma a la lista
    frames_data.append(frame_data)

# El número de planetas es el número de líneas en cada bloque
# Lo calculamos del primer bloque
nplanets = len(frames_data[0])


# Creación de la animación/gráfico
# ========================================
# Crea los objetos figure y axis
fig, ax = plt.subplots()

# Define el rango de los ejes
ax.axis("equal")  # Misma escala para ejes X e Y
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

# Si solo se ha dado un radio para todos los planetas, conviértelo a una
# lista con todos los elementos iguales
if not hasattr(planet_radius, "__iter__"):
    planet_radius = planet_radius*np.ones(nplanets)
# En caso contrario, comprueba que el nº de radios coincide con el
# nº de planetas y devuelve error en caso contrario
else:
    if not nplanets == len(planet_radius):
        raise ValueError(
                "El número de radios especificados no coincide con el número "
                "de planetas")

# Si solo se ha dado un color para todos los planetas, convíertelo a una
# lista con todos los elementos iguales
if not hasattr(planet_colors, "__iter__") or isinstance(planet_colors, str):
    planet_colors = [planet_colors]*nplanets
elif len(planet_colors) < nplanets:
    # Repite la secuencia de colores si hay menos de planetas.
    planet_colors = (planet_colors *
                     ((nplanets // len(planet_colors)) + 1))[:nplanets]

# Representa el primer fotograma
# Pinta un punto en la posición de cada planeta y guarda el objeto asociado
# al punto en una lista
planet_points = list()
planet_trails = list()
for planet_pos, radius, color in zip(frames_data[0], planet_radius, planet_colors):
    x, y = planet_pos
    planet_point = Circle((x, y), radius, facecolor=color, edgecolor="none")
    ax.add_artist(planet_point)
    planet_points.append(planet_point)

    # Inicializa las estelas (si especificado en los parámetros)
    if show_trail:
        planet_trail, = ax.plot(
                x, y, "-", linewidth=trail_width,
                color=color)
        planet_trails.append(planet_trail)
 
# Función que actualiza la posición de los planetas en la animación 
def update(j_frame, frames_data, planet_points, planet_trails, show_trail):
    # Actualiza la posición del correspondiente a cada planeta
    for j_planet, planet_pos in enumerate(frames_data[j_frame]):
        x, y = planet_pos
        planet_points[j_planet].center = (x, y)

        if show_trail:
            xs_old, ys_old = planet_trails[j_planet].get_data()
            xs_new = np.append(xs_old, x)
            ys_new = np.append(ys_old, y)

            planet_trails[j_planet].set_data(xs_new, ys_new)

    return planet_points + planet_trails

def init_anim():
    # Clear trails
    if show_trail:
        for j_planet in range(nplanets):
            planet_trails[j_planet].set_data(list(), list())

    return planet_points + planet_trails

# Calcula el nº de frames
nframes = len(frames_data)

# Si hay más de un instante de tiempo, genera la animación
if nframes > 1:
    # Info sobre FuncAnimation: https://matplotlib.org/stable/api/animation_api.html
    animation = FuncAnimation(
            fig, update, init_func=init_anim,
            fargs=(frames_data, planet_points, planet_trails, show_trail),
            frames=len(frames_data), blit=True, interval=interval)

    # Muestra por pantalla o guarda según parámetros
    if save_to_file:
        animation.save("{}.mp4".format(file_out), dpi=dpi)
    else:
        plt.show()
# En caso contrario, muestra o guarda una imagen
else:
    # Muestra por pantalla o guarda según parámetros
    if save_to_file:
        fig.savefig("{}.pdf".format(file_out))
    else:
        plt.show()


## Resultados.

Las trayectorias se muestran en la carpeta en vídeos llamados adap.mp4 y fija.mp4. El experimento realizado tiene una velocidad inicial del cohete demasiado baja, lo cual impide que llegue a orbitar. Cambiando las condiciones iniciales se puede conseguir un mejor resultado.

El método fijo tardó 28801 iteraciones mientras que el adaptativo utilizó más, 57602. El error acumulado en H' es minúsculo en ambos casos: $6.07552·10^{-19}$ y $5.79182·10^{-22}$ respectivamente, lo cual indica que permanece constante.